# Banking Grammar Assistant — Qwen3-0.6B + Pandas vs Spark

This notebook has **two goals**:

1. Build and fine-tune a small banking grammar assistant.
2. Compare **Pandas vs Apache Spark on the same data-processing task**, then use each tool where it makes sense.

The important engineering point is that **Spark is not automatically faster than Pandas**. On one Kaggle machine and tens of thousands of rows, Pandas can easily win because Spark pays JVM, scheduling, serialization, and shuffle overhead. Spark becomes attractive when the data grows beyond comfortable single-process memory, when joins/dedup/aggregations become large, or when the same pipeline is moved to a real cluster.

### Pipeline

```text
Raw corpora
   │
   ├── Pandas benchmark ─────┐
   │                         ├── same transformation / same output
   └── Spark benchmark ──────┘
                 │
                 ▼
      Hybrid data-engineering pipeline
      ├── Spark: large tabular cleaning, dedupe, union, split, Parquet
      └── Python/Pandas: custom difflib + synthetic corruption on bounded samples
                 │
                 ▼
        LLaMA-Factory SFT
        Qwen3-0.6B on 1×T4
                 │
                 ▼
             Evaluation
```

**Why hybrid?** The custom grammar-diff code is Python-heavy (`difflib`, regexes, Python dict/list logic). Forcing every small custom operation through Spark can make the notebook slower, not more "big data". We use Spark where its execution engine is actually useful.


## 0. Environment

In [ ]:
# Kaggle setup. If LLaMA-Factory already exists, the clone step is skipped.
import os, shutil, subprocess, re as _re
from pathlib import Path

# Pinned on purpose. LLaMA-Factory `main` and the latest transformers drift
# apart: on 2026-11-20 they broke this notebook twice (transformers 5.x dropped
# `warmup_ratio`; transformers>=5.16 passes `every_n_layers` to a
# gradient-checkpointing hook LLaMA-Factory overrides without that argument).
LF_COMMIT = "b4e4d3fcb2a5a13ce6b9431d395e3939f84b8ec1"

if not Path("LLaMA-Factory").exists():
    subprocess.run(["git", "init", "-q", "LLaMA-Factory"], check=True)
    subprocess.run(["git", "-C", "LLaMA-Factory", "fetch", "-q", "--depth", "1",
                    "https://github.com/hiyouga/LLaMA-Factory.git", LF_COMMIT], check=True)
    subprocess.run(["git", "-C", "LLaMA-Factory", "checkout", "-q", "FETCH_HEAD"], check=True)

%cd LLaMA-Factory
%pip install -q -e ".[torch,metrics]"
%cd ..

# Spark SQL extra brings Arrow/Pandas integration dependencies.
%pip install -q "pyspark[sql]==4.0.2" datasets "transformers==5.15.1" accelerate sentencepiece pyyaml psutil pandas pyarrow

In [ ]:
# Spark 4 requires a JVM >= 17. Kaggle images normally ship one, but a silent
# JVM mismatch surfaces as an opaque Py4J crash much later, so check it now.
#   - root + apt-get (Kaggle / Colab): install OpenJDK 17 system-wide
#   - otherwise (local machine, no sudo): download a Temurin 17 JRE into ~/.jdk

import shutil, subprocess, os, sys, glob, re as _re


def java_major():
    exe = shutil.which("java")
    if not exe:
        return None
    out = subprocess.run([exe, "-version"], capture_output=True, text=True)
    m = _re.search(r'version "(\d+)', out.stderr + out.stdout)
    return int(m.group(1)) if m else None


def use_java_home(home):
    os.environ["JAVA_HOME"] = home
    os.environ["PATH"] = os.path.join(home, "bin") + os.pathsep + os.environ["PATH"]


jv = java_major()
print("Java major version:", jv)

if jv is None or jv < 17:
    if os.geteuid() == 0 and shutil.which("apt-get"):
        print("Installing OpenJDK 17 via apt ...")
        subprocess.run(
            "apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless",
            shell=True, check=True,
        )
        use_java_home("/usr/lib/jvm/java-17-openjdk-amd64")
    else:
        # No root: user-local JRE, downloaded once and reused on later runs.
        jdk_dir = os.path.expanduser("~/.jdk")
        found = sorted(glob.glob(os.path.join(jdk_dir, "jdk-17*")))
        if found:
            home = found[-1]
        else:
            print("No root access; downloading a Temurin 17 JRE into ~/.jdk ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "install-jdk"], check=True)
            import jdk
            home = jdk.install("17", jre=True, path=jdk_dir)
        use_java_home(home)
    print("Java major version now:", java_major())

In [ ]:
import os, json, re, random, difflib, unicodedata, time, gc
from pathlib import Path
from collections import Counter

import pandas as pd
import psutil
import torch

random.seed(13)

ROOT      = Path("gec_bank")
DATA_DIR  = ROOT / "data"
CFG_DIR   = ROOT / "configs"
SAVE_DIR  = ROOT / "saves"
for d in (DATA_DIR, CFG_DIR, SAVE_DIR):
    d.mkdir(parents=True, exist_ok=True)

LF_DIR = Path("LLaMA-Factory")
BASE_MODEL = "Qwen/Qwen3-0.6B"

# ---- run mode -----------------------------------------------------------
# SMOKE rehearses the whole pipeline in minutes so a broken cell surfaces
# before a multi-hour GPU commit. Set to False for the real run.
SMOKE = False

if SMOKE:
    N_COEDIT_GEC   = 2_000
    N_C4_200M      = 3_000
    N_WI_LOCNESS   = 2_000
    N_LANG8        = 1_500
    N_BANK_NOOP    = 1_500
    N_BANK_GEC     = 800
    BENCH_LIMIT    = 20_000     # rows of raw CoEdIT fed to the engine benchmark
    CFPB_LIMIT     = 200_000    # narratives read from the CFPB CSV
    EDGAR_DOCS     = 60
    BANK_SAMPLE_N  = 5_000
else:
    # MEASURED: the first full run trained on 6,000 banking GEC examples out
    # of 117,261 correction examples -- 5.1% banking for a banking product. It
    # scored F0.5 0.639 on banking but only 0.267 on general web text, and the
    # original eval reported the general number. Rebalance toward the domain
    # that is actually being sold, and cut the CoEdIT/C4 share: their gold
    # labels contain stylistic rewrites, which teach the model to change
    # content rather than fix grammar.
    # eval_loss plateaued INSIDE epoch 1 (last delta -0.0002), so a second
    # epoch buys nothing -- the model already fit this distribution. The lever
    # is a different distribution: reweighted error types (section 7) and more
    # unique source sentences, drawn from the 4M available rather than reusing
    # the same ones. General stays put; cutting it last time improved general
    # scores, so more of it is not the answer either.
    # Every correction example so far was synthetic: banking from our own nine
    # corrupt() functions, C4-200M from an automatic tagger. That is the real
    # credibility gap -- and on JFLEG (human-annotated learner English, scored
    # with the corpus tokenisation) it shows: 50.5 GLEU for this model vs 55.6
    # for coedit-large, which WAS trained on real learner corpora.
    # W&I+LOCNESS is the BEA-2019 corpus and Lang-8 is real learner
    # corrections, so they buy generalisation to errors nobody scripted.
    N_COEDIT_GEC   = 18_000
    N_C4_200M      = 30_000
    N_WI_LOCNESS   = 35_000
    N_LANG8        = 25_000
    # noop_rate on clean banking text rose 0.138 -> 0.195 when the model was
    # trained to be bolder on hard error types. More clean examples is the
    # direct counterweight: they teach it that "[]" is often the right answer.
    N_BANK_NOOP    = 24_000
    N_BANK_GEC     = 70_000
    BENCH_LIMIT    = None       # None = full CoEdIT train split
    CFPB_LIMIT     = None       # None = every narrative
    EDGAR_DOCS     = 400
    BANK_SAMPLE_N  = 20_000

NOOP_RATIO_GEN = 0.12

_T0 = time.time()

def stage(msg):
    """Timestamped marker. A stalled run is only diagnosable if the log says
    which phase it stalled in and how long the previous phase took."""
    print(f"[stage +{time.time() - _T0:7.1f}s] {msg}", flush=True)

print("SMOKE run" if SMOKE else "FULL run")
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {p.total_memory / 1024**3:.1f} GiB")

# Fail fast on an incompatible GPU. Kaggle hands out a P100 (sm_60) unless the
# accelerator is pinned, and torch 2.10+cu128 ships no sm_60 kernels -- which
# otherwise surfaces minutes later as an opaque
# "CUDA error: no kernel image is available for execution on the device".
if torch.cuda.is_available():
    arches = {int(a.split("_")[1]) for a in torch.cuda.get_arch_list() if a.startswith("sm_")}
    for i in range(torch.cuda.device_count()):
        major, minor = torch.cuda.get_device_capability(i)
        cap = major * 10 + minor
        if cap not in arches:
            raise RuntimeError(
                f"GPU {i} ({torch.cuda.get_device_name(i)}) is sm_{cap}, but torch "
                f"{torch.__version__} only has kernels for {sorted(arches)}. "
                "Set the notebook accelerator to 'GPU T4 x2' and re-run."
            )
    print("GPU arch check passed. torch kernels:", sorted(arches))

print("CPU cores:", os.cpu_count())
print("RAM:", round(psutil.virtual_memory().total / 1024**3, 1), "GiB")

### Start Spark

This is **Spark local mode** on the Kaggle VM. It uses the Spark DataFrame/SQL engine and all available CPU cores, but it is still one machine. The same DataFrame code can later move to a multi-node Spark cluster.


In [ ]:
# In local mode the driver IS the executor, and its heap is fixed when the JVM
# launches -- .config("spark.driver.memory") after the fact is ignored. Kaggle
# gives ~31 GiB of RAM, so take a real slice of it before pyspark starts.
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 8g pyspark-shell"

from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder
    .appName("QwenBankGrammarPipeline")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", str(max(4, min(16, os.cpu_count() or 4))))
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Parallelism:", spark.sparkContext.defaultParallelism)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))


## 1. Protected terminology

In [ ]:
PROTECTED_TERMS = {
    # instruments & identifiers
    "IBAN", "SWIFT", "BIC", "ACH", "SEPA", "IFSC", "MICR", "CUSIP", "ISIN", "LEI",
    "APR", "APY", "EAR", "LIBOR", "SOFR", "EURIBOR", "CBE", "FRB", "FDIC", "OCC",
    "KYC", "AML", "CDD", "EDD", "PEP", "SAR", "CTR", "UDAAP", "TILA", "RESPA",
    "ACH", "POS", "ATM", "PIN", "CVV", "OTP", "SLA", "NDA", "PSD2", "GDPR",
    "LTV", "DTI", "FICO", "NPV", "IRR", "EBITDA", "YTD", "QoQ", "YoY",
    "Basel", "IFRS", "GAAP", "CRS", "FATCA",
    # legalese that reads like an error but is not
    "hereinafter", "heretofore", "thereunder", "thereof", "herein", "hereby",
    "notwithstanding", "aforementioned", "whereas", "pursuant", "thereto",
    "indemnify", "indemnification", "forbearance", "novation", "subrogation",
    "escrow", "lien", "encumbrance", "covenant", "accrual", "amortization",
    "per annum", "pro rata", "force majeure", "bona fide", "inter alia",
}

# Regexes for things that must survive verbatim
PROTECTED_PATTERNS = [
    re.compile(r"\b[A-Z]{2,}\b"),                              # any all-caps acronym
    re.compile(r"[$€£¥]\s?[\d,]+(?:\.\d+)?"),                  # $1,250.00
    re.compile(r"\b(?:USD|EUR|GBP|EGP|AED|SAR|JPY)\s?[\d,]+"),  # EGP 1,250,000
    re.compile(r"\b\d+(?:\.\d+)?\s?%"),                       # 4.75 %
    re.compile(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b"),                # dates
    re.compile(r"\bSection\s+\d+"),                            # Section 4.2
    re.compile(r"\bClause\s+\d+"),
    re.compile(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\s+Agreement\b"),  # defined terms
]

def strip_punct(tok):
    """Split a whitespace token into (lead_punct, core, trail_punct)."""
    m = re.match(r"^([^\w]*)(.*?)([^\w]*)$", tok, flags=re.UNICODE)
    return m.group(1), m.group(2), m.group(3)

# multi-word terms need span patterns, or the corruptor will split them
# ("per annum" -> "per the annum"). Auto-generated so the term list stays simple.
PROTECTED_PATTERNS += [
    re.compile(r"\b" + re.escape(t) + r"\b", re.I)
    for t in PROTECTED_TERMS if " " in t
]

_PROTECTED_LOWER = {t.lower() for t in PROTECTED_TERMS}

def is_protected(token, sentence=None):
    _, core, _ = strip_punct(token)
    if not core:
        return True
    if core.lower() in _PROTECTED_LOWER:      # case-insensitive: "Pursuant" == "pursuant"
        return True
    if core.isupper() and len(core) >= 2:
        return True
    if any(ch.isdigit() for ch in core):
        return True
    if sentence:
        for pat in PROTECTED_PATTERNS:
            for m in pat.finditer(sentence):
                if core in m.group(0):
                    return True
    return False

def free(tokens, i, sentence):
    """Safe to insert *after* token i? Must not split a protected multi-token span."""
    if is_protected(tokens[i], sentence):
        return False
    return i + 1 >= len(tokens) or not is_protected(tokens[i + 1], sentence)

print(f"{len(PROTECTED_TERMS)} protected terms, {len(PROTECTED_PATTERNS)} patterns")

## 2. Compact edit format

In [ ]:
WORD_RE = re.compile(r"\S+")

def toks(s):
    return WORD_RE.findall(s)

def derive_edits(src, tgt):
    """Word-level diff -> compact edit list.

    Occurrence index `n` is counted in *substring* space, not word space, because
    apply_edits resolves spans with str.find(). Getting this wrong silently breaks
    any sentence containing a repeated word ("the" also occurs inside "there").
    """
    spans = [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(src)]
    a = [s[0] for s in spans]
    b = toks(tgt)
    edits = []
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(a=a, b=b, autojunk=False).get_opcodes():
        if tag == "equal":
            continue
        if tag == "insert":
            # anchor to an adjacent word: span anchors survive tokenizer drift,
            # integer positions do not
            if i1 > 0:
                k, pre = i1 - 1, True
            elif i1 < len(a):
                k, pre = i1, False
            else:
                return None
            cs, ce = spans[k][1], spans[k][2]
            o = src[cs:ce]
            ins = " ".join(b[j1:j2])
            c = (o + " " + ins) if pre else (ins + " " + o)
        else:
            cs, ce = spans[i1][1], spans[i2 - 1][2]
            o = src[cs:ce]
            c = " ".join(b[j1:j2])
        edits.append({"o": o, "c": c, "n": src[:cs].count(o) + 1})
    return edits or None

def apply_edits(src, edits):
    """Inverse of derive_edits. Used to validate every training pair round-trips."""
    out = src
    for e in edits:
        o, c, n = e["o"], e["c"], e.get("n", 1)
        idx, start = -1, 0
        for _ in range(n):
            idx = out.find(o, start)
            if idx == -1:
                return None
            start = idx + 1
        out = out[:idx] + c + out[idx + len(o):]
    return re.sub(r"\s+", " ", out).strip()

def roundtrips(src, tgt):
    e = derive_edits(src, tgt)
    if e is None:
        return None
    got = apply_edits(src, e)
    norm = lambda s: re.sub(r"\s+", " ", s).strip()
    return e if got is not None and norm(got) == norm(tgt) else None

# sanity check
_s = "The account holder were notified of the changes to there statement."
_t = "The account holder was notified of the changes to their statement."
print(json.dumps(roundtrips(_s, _t), ensure_ascii=False))
print(apply_edits(_s, roundtrips(_s, _t)))

### Spark wrapper around the Python diff logic

`roundtrips()` is custom Python logic. Spark cannot optimize `difflib.SequenceMatcher`, so we expose one boundary to Python and return JSON.

For the benchmark we deliberately use the **same `roundtrips()` implementation** in Pandas and Spark so the comparison is about execution engines, not different algorithms.


In [ ]:
@F.udf(returnType=T.StringType())
def roundtrips_udf(src, tgt):
    if src is None or tgt is None:
        return None
    edits = roundtrips(src, tgt)
    if edits is None:
        return None
    return json.dumps(edits, ensure_ascii=False, separators=(",", ":"))

EDIT_SCHEMA = T.ArrayType(
    T.StructType([
        T.StructField("o", T.StringType(), False),
        T.StructField("c", T.StringType(), False),
        T.StructField("n", T.IntegerType(), False),
    ])
)

test_df = spark.createDataFrame(
    [
        (
            "The account holder were notified of the changes to there statement.",
            "The account holder was notified of the changes to their statement.",
        ),
        (
            "This sentence is already correct.",
            "This sentence is already correct.",
        ),
    ],
    ["src", "tgt"],
)

test_df.withColumn("edits", roundtrips_udf("src", "tgt")).show(truncate=False)


## 3. Prompt template

In [ ]:
INSTRUCTION = (
    "Correct grammar, spelling, and punctuation. Do not change meaning, tone, "
    "financial terminology, defined terms, figures, or legal phrasing. "
    "Reply with a JSON array of edits, or [] if the text is correct."
)

def to_alpaca(src, edits, tag):
    return {
        "instruction": INSTRUCTION,
        "input": src,
        "output": json.dumps(edits, ensure_ascii=False, separators=(",", ":")),
        "_tag": tag,
    }

print(json.dumps(to_alpaca(_s, roundtrips(_s, _t), "demo"), indent=2)[:400])

## 4. Pandas vs Spark — fair benchmark

We load **one raw CoEdIT sample**, save it once to Parquet, then make both engines read the same file and run the same logical pipeline:

1. keep `task == "gec"`
2. remove the natural-language instruction prefix
3. trim target
4. remove empty / long examples
5. run `roundtrips`
6. keep valid examples with at most 8 edits

Two details matter:

- Spark is **lazy**. Timing a transformation without an action is fake, so the Spark benchmark ends with `count()`.
- On this Kaggle VM, Spark is local. A local benchmark measures local overhead/throughput, not cluster scalability.


In [ ]:
stage("loading CoEdIT")

from datasets import load_dataset

coedit_hf = load_dataset("grammarly/coedit", split="train")

COEDIT_RAW = DATA_DIR / "coedit_raw.parquet"

# Dataset.select() takes *row indices*; column projection is select_columns().
coedit_hf.select_columns(["task", "src", "tgt"]).to_parquet(str(COEDIT_RAW))

print("Rows:", len(coedit_hf))
print("Parquet:", COEDIT_RAW)

In [ ]:
def tree_rss_mb():
    # RSS of Python + child processes, including Spark's JVM.
    p = psutil.Process()
    procs = [p] + p.children(recursive=True)
    
    total = 0
    for proc in procs:
        try:
            total += proc.memory_info().rss
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return total / 1024**2


# Both engines must apply *character-identical* string rules, or the row counts
# diverge and the benchmark compares two different pipelines. Python's \s and
# Java's \s disagree on Unicode spaces, so spell the class out explicitly.
WS       = r"[ \t\n\r\f\x0B]"
RE_TRIM  = rf"^{WS}+|{WS}+$"
RE_PREFIX = rf"^[^:]*:{WS}*"


def pandas_coedit_pipeline(path, limit=None):
    pdf = pd.read_parquet(path)
    if limit is not None:
        pdf = pdf.head(limit).copy()

    pdf = pdf.loc[pdf["task"].eq("gec"), ["src", "tgt"]].copy()
    pdf["src"] = (
        pdf["src"]
        .str.replace(RE_PREFIX, "", regex=True)
        .str.replace(RE_TRIM, "", regex=True)
    )
    pdf["tgt"] = pdf["tgt"].str.replace(RE_TRIM, "", regex=True)

    mask = (
        pdf["src"].ne("")
        & pdf["tgt"].ne("")
        & pdf["src"].str.len().le(400)
    )
    pdf = pdf.loc[mask].copy()

    pdf["edits_obj"] = [
        roundtrips(s, t) for s, t in zip(pdf["src"], pdf["tgt"])
    ]
    pdf = pdf[
        pdf["edits_obj"].map(lambda x: x is not None and len(x) <= 8)
    ].copy()

    pdf["edits"] = pdf["edits_obj"].map(
        lambda x: json.dumps(x, ensure_ascii=False, separators=(",", ":"))
    )
    return pdf[["src", "tgt", "edits"]]


def spark_coedit_pipeline(path, limit=None):
    sdf = spark.read.parquet(str(path))
    if limit is not None:
        sdf = sdf.limit(limit)

    sdf = (
        sdf
        .filter(F.col("task") == "gec")
        .select(
            F.regexp_replace(
                F.regexp_replace("src", RE_PREFIX, ""), RE_TRIM, ""
            ).alias("src"),
            F.regexp_replace("tgt", RE_TRIM, "").alias("tgt"),
        )
        .filter(
            (F.length("src") > 0)
            & (F.length("tgt") > 0)
            & (F.length("src") <= 400)
        )
        .withColumn("edits_json", roundtrips_udf("src", "tgt"))
        .withColumn("edits_obj", F.from_json("edits_json", EDIT_SCHEMA))
        .filter(
            F.col("edits_obj").isNotNull()
            & (F.size("edits_obj") <= 8)
        )
        .select("src", "tgt", F.col("edits_json").alias("edits"))
    )
    return sdf

In [ ]:
stage("pandas-vs-spark benchmark")

# BENCH_LIMIT is set by the run-mode block in section 0.
bench = []

gc.collect()
m0 = tree_rss_mb()
t0 = time.perf_counter()
pd_result = pandas_coedit_pipeline(COEDIT_RAW, BENCH_LIMIT)
pd_sec = time.perf_counter() - t0
m1 = tree_rss_mb()

bench.append({
    "engine": "pandas",
    "seconds": pd_sec,
    "output_rows": len(pd_result),
    "rss_delta_mb": max(0.0, m1 - m0),
})

# Spark is lazy: count() forces execution and belongs inside the timed region.
gc.collect()
m0 = tree_rss_mb()
t0 = time.perf_counter()
sp_result = spark_coedit_pipeline(COEDIT_RAW, BENCH_LIMIT)
sp_rows = sp_result.count()
sp_sec = time.perf_counter() - t0
m1 = tree_rss_mb()

bench.append({
    "engine": "spark-local",
    "seconds": sp_sec,
    "output_rows": sp_rows,
    "rss_delta_mb": max(0.0, m1 - m0),
})

bench_df = pd.DataFrame(bench)
display(bench_df)
# display() renders to the notebook only; a batch run's log would otherwise
# have no record of the benchmark at all.
print(bench_df.to_string(index=False))

# A mismatch invalidates the comparison, but it must not kill a long GPU run:
# report it loudly and let the training stages continue.
if len(pd_result) == sp_rows:
    print("Row counts match -> the two engines ran an equivalent pipeline.")
else:
    print(
        f"WARNING: output rows differ (pandas={len(pd_result)}, spark={sp_rows}). "
        "The timings below compare pipelines that are not equivalent; "
        "fix the transformation before quoting this benchmark."
    )

winner = bench_df.sort_values("seconds").iloc[0]["engine"]
print("Local wall-time winner:", winner)

### How to interpret the result

**Do not conclude "Pandas is better" or "Spark is better" from one number.**

For this notebook:

- **Pandas** is usually the better choice for small/medium in-memory data and Python-heavy row logic.
- **Spark** is the better engineering choice for large CSV/Parquet corpora, big dedupes/joins/aggregations, data that does not fit comfortably in one Python process, or a pipeline you expect to move to a cluster.
- A Python UDF is one of Spark's weaker paths because Spark has to cross from the JVM execution engine into Python. Prefer Spark-native functions for cleaning/filtering whenever possible.
- `toPandas()` collects everything to the driver. We only use it on bounded/dev samples.

So the final pipeline below is deliberately **hybrid**, not ideological.


## 5. General GEC data

In [ ]:
# CoEdIT is moderate-sized and the expensive transform is custom Python/difflib,
# so we use the Pandas result for the actual local Kaggle pipeline.
coedit_pairs = [
    (row.src, row.tgt, json.loads(row.edits))
    for row in pd_result.itertuples(index=False)
]

random.shuffle(coedit_pairs)
coedit_pairs = coedit_pairs[:N_COEDIT_GEC]
print(f"CoEdIT usable pairs: {len(coedit_pairs)}")


In [ ]:
stage("streaming C4-200M")

# C4-200M. The original `liweili/c4_200m` is a *script* dataset, and datasets>=4
# refuses to execute dataset scripts, so it silently produced zero pairs. This
# mirror ships the same input/output GEC pairs as plain parquet. The original is
# kept as a fallback for environments still on datasets<4.
C4_SOURCES = [
    "512duncanl/c4_200m_cleaned_365k",
    "liweili/c4_200m",
]

c4 = None
for repo in C4_SOURCES:
    try:
        c4 = load_dataset(repo, split="train", streaming=True)
        print("C4-200M source:", repo)
        break
    except Exception as ex:
        print(f"  {repo} unavailable: {ex}")

c4_pairs = []

if c4 is None:
    print("No C4-200M source reachable; continuing without it.")
else:
    scanned = 0
    for row in c4:
        # Hard bound on streaming work: without it a low acceptance rate turns
        # this cell into an open-ended download.
        scanned += 1
        if scanned > N_C4_200M * 25:
            print(f"Stopping after scanning {scanned:,} rows.")
            break

        s = (row.get("input") or "").strip()
        t = (row.get("output") or "").strip()

        if not s or not t or len(s) > 300 or s == t:
            continue

        e = roundtrips(s, t)
        if e is not None and 1 <= len(e) <= 5:
            c4_pairs.append((s, t, e))

        if len(c4_pairs) >= N_C4_200M:
            break

print(f"C4-200M pairs: {len(c4_pairs)}")
if not c4_pairs:
    print("WARNING: no C4 pairs -- general-domain GEC coverage will be thin.")

In [ ]:
stage("real human-error corpora (W&I+LOCNESS, Lang-8)")

# Both are genuine human-annotated learner corrections, unlike C4-200M (an
# automatic tagger) and the banking set (our own corrupt()). W&I+LOCNESS is the
# BEA-2019 shared-task corpus; Lang-8 is noisier but large. Neither is a script
# dataset, so datasets>=4 can load them.
REAL_SOURCES = [
    ("wi_locness", "512duncanl/wi_locness_detokenized", "input", "output", N_WI_LOCNESS),
    ("lang8",      "rahuln2002/GED-lang8-cleaned",      "0",     "1",      N_LANG8),
]

real_pairs = {}

for tag, repo, ci, co, want in REAL_SOURCES:
    got = []
    try:
        ds = load_dataset(repo, split="train", streaming=True)
        scanned = 0
        for row in ds:
            scanned += 1
            if scanned > want * 30:
                print(f"  {tag}: stopping after {scanned:,} rows")
                break
            s = (row.get(ci) or "").strip()
            t = (row.get(co) or "").strip()
            if not s or not t or s == t or len(s) > 300:
                continue
            e = roundtrips(s, t)
            if e is not None and 1 <= len(e) <= 5:
                got.append((s, t, e))
            if len(got) >= want:
                break
    except Exception as ex:
        print(f"  {tag} unavailable: {ex}")
    real_pairs[tag] = got
    print(f"{tag}: {len(got):,} pairs")

WI_PAIRS    = real_pairs.get("wi_locness", [])
LANG8_PAIRS = real_pairs.get("lang8", [])

n_real = len(WI_PAIRS) + len(LANG8_PAIRS)
if not n_real:
    print("WARNING: no real human-error data loaded; the model will only ever "
          "see errors that a script generated.")

## 6. Banking corpus — Spark for the large tabular stage

This is where Spark earns its place.

- Seed sentences are tiny.
- CFPB complaints can become large, so CSV read, explode, filter, and dedupe are Spark-native.
- EDGAR is streamed from Hugging Face in a bounded sample, then unioned into Spark.
- We only collect a bounded sample back to Python for synthetic corruption.


In [ ]:
SEED_BANKING = [
    "The Bank shall hereinafter indemnify the Account Holder against any loss arising from unauthorised transactions.",
    "Interest accrues daily at a rate of 4.75% per annum and is credited to the account on the last business day of each month.",
    "Please note that the APR disclosed in Section 4.2 is variable and may change without prior notice.",
    "Your outstanding balance of EGP 1,250,000.00 is due on 15/03/2026.",
    "We were unable to process your transfer because the IBAN provided did not match our records.",
    "Notwithstanding the foregoing, the Customer remains liable for all charges incurred prior to the date of termination.",
    "Pursuant to our KYC obligations, we require an updated proof of address within thirty days.",
    "The Bank reserves the right to close any account that remains dormant for a period exceeding twelve consecutive months.",
    "Funds transferred via SWIFT may take up to three business days to clear.",
    "This Agreement constitutes the entire understanding between the parties and supersedes all prior arrangements.",
    "Your application for a mortgage was declined because the loan-to-value ratio exceeded our lending threshold.",
    "We have credited your account with a provisional refund while we investigate the disputed charge.",
    "Failure to maintain the minimum balance will result in a monthly maintenance fee being applied.",
    "The Customer acknowledges that the Bank is not liable for delays caused by force majeure events.",
    "A late payment fee of 2% of the outstanding amount will be charged after the grace period expires.",
    "Please contact our Customer Care team if you did not authorise this transaction.",
    "Enhanced due diligence is required for accounts associated with politically exposed persons.",
    "The exchange rate applied is the prevailing rate at the time the transaction is processed.",
    "We regret to inform you that your request for a credit limit increase has not been approved at this time.",
    "All statements are available in electronic form through our secure online banking portal.",
]


In [ ]:
stage("building banking corpus")

def spark_clean_sentence_df(df):
    return (
        df
        .select(F.trim(F.regexp_replace("text", r"\s+", " ")).alias("text"))
        .filter(F.col("text").isNotNull())
        .filter(F.length("text") > 0)
        .withColumn("word_count", F.size(F.split("text", r"\s+")))
        .filter(F.col("word_count").between(6, 45))
        .filter(F.col("text").rlike(r"^[A-Z]"))
        .filter(F.col("text").rlike(r"[.?!]$"))
        # CFPB narratives are redacted with runs of X ("XXXX", "XX/XX/XXXX").
        # Training a grammar checker on redaction masks teaches it nothing good.
        .filter(~F.col("text").rlike(r"(?i)(http|www\.|@|x{3,}|\bxx\b|_{2,}|\|)"))
        .dropDuplicates(["text"])
        .select("text")
    )


seed_df = spark.createDataFrame([(s,) for s in SEED_BANKING], ["text"])

EMPTY_TEXT_DF = spark.createDataFrame([], "text string")


def find_complaints_csv():
    """CFPB CSV arrives as an attached Kaggle dataset under /kaggle/input."""
    for root in (Path("/kaggle/input"), Path(".")):
        if not root.exists():
            continue
        hits = sorted(root.glob("**/complaints.csv"))
        if hits:
            return hits[0]
    return None


cfpb_csv = find_complaints_csv()
cfpb_sentences = EMPTY_TEXT_DF

if cfpb_csv is None:
    print("complaints.csv not found — CFPB stage skipped. "
          "Attach a CFPB complaints dataset to include it.")
else:
    print("CFPB source:", cfpb_csv)
    cfpb_raw = (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(str(cfpb_csv))
    )

    # Header spelling varies between CFPB exports and Kaggle re-uploads.
    narrative_col = next(
        (c for c in cfpb_raw.columns if "narrative" in c.lower()), None
    )

    if narrative_col is None:
        print("No narrative column found. Columns:", cfpb_raw.columns[:25])
    else:
        print("Narrative column:", repr(narrative_col))
        narratives = (
            cfpb_raw
            .select(F.col(narrative_col).alias("narrative"))
            .filter(F.col("narrative").isNotNull())
        )
        if CFPB_LIMIT is not None:
            narratives = narratives.limit(CFPB_LIMIT)

        cfpb_sentences = narratives.select(
            F.explode(F.split("narrative", r"(?<=[.!?])\s+")).alias("text")
        )


edgar_sentences = []
try:
    # eloukas/edgar-corpus is also a script dataset, but the repo stores plain
    # per-year JSONL alongside it, which loads without executing any script.
    ed = load_dataset(
        "json",
        data_files="hf://datasets/eloukas/edgar-corpus/2020/train.jsonl",
        split="train",
        streaming=True,
    )
    for i, row in enumerate(ed):
        body = row.get("section_1") or ""
        edgar_sentences.extend(
            s.strip() for s in re.split(r"(?<=[.!?])\s+", body)[:40] if s.strip()
        )
        if i >= EDGAR_DOCS:
            break
except Exception as ex:
    print("EDGAR unavailable:", ex)

print("EDGAR sentences:", len(edgar_sentences))

edgar_df = (
    spark.createDataFrame([(s,) for s in edgar_sentences], ["text"])
    if edgar_sentences
    else EMPTY_TEXT_DF
)

bank_df = spark_clean_sentence_df(
    seed_df.unionByName(cfpb_sentences).unionByName(edgar_df)
).cache()

print("Clean banking sentences:", bank_df.count())
bank_df.show(10, truncate=120)

In [ ]:
bank_profile = bank_df.select(
    F.count("*").alias("rows"),
    F.avg(F.length("text")).alias("avg_chars"),
    F.avg(F.size(F.split("text", r"\s+"))).alias("avg_words"),
    F.max(F.size(F.split("text", r"\s+"))).alias("max_words"),
)

bank_profile.show(truncate=False)

# Bounded sample back to Python for custom synthetic corruption.
# BANK_SAMPLE_N is set by the run-mode block in section 0.
sample_n = max(N_BANK_NOOP, N_BANK_GEC * 3, BANK_SAMPLE_N)

# NOT orderBy(rand()).limit(n): that compiles to takeOrdered, which builds an
# n-row priority queue *in the driver* over every input row. It survived at
# n=20,000 and blew the driver heap at n=135,000 on 4M sentences. sample() is
# a per-partition filter -- no global ordering, no driver-side accumulation.
bank_total = bank_df.count()
frac = min(1.0, (sample_n * 1.4) / max(bank_total, 1))

BANK_SENTS = [
    r["text"]
    for r in bank_df.sample(withReplacement=False, fraction=frac, seed=13)
                    .limit(sample_n)
                    .collect()
]
print(f"sampled {len(BANK_SENTS):,} of {bank_total:,} (fraction {frac:.4f})")

# Hold sentences out BEFORE any of them reach the training set. Scoring the
# model on banking text it trained on would flatter it.
N_BANK_HELD = 3_000 if not SMOKE else 500
BANK_EVAL_SENTS = BANK_SENTS[:N_BANK_HELD]
BANK_SENTS      = BANK_SENTS[N_BANK_HELD:]

print("Collected banking sample:", len(BANK_SENTS), "train /",
      len(BANK_EVAL_SENTS), "held out for eval")

## 7. Error injection — keep custom Python logic

In [ ]:
AGREEMENT = {"is":"are","are":"is","was":"were","were":"was","has":"have","have":"has",
             "does":"do","do":"does","wasn't":"weren't","weren't":"wasn't",
             "isn't":"aren't","aren't":"isn't","hasn't":"haven't","haven't":"hasn't"}

CONFUSIONS = {"their":"there","there":"their","they're":"their","its":"it's","it's":"its",
              "then":"than","than":"then","affect":"effect","effect":"affect",
              "principal":"principle","principle":"principal","ensure":"insure",
              "advice":"advise","advise":"advice","accept":"except","except":"accept",
              "lose":"loose","loose":"lose","complement":"compliment",
              "your":"you're","you're":"your","to":"too","too":"to","whose":"who's"}

PREPS = ["of","for","to","in","on","at","with","from","by","about","under","within"]

TENSE = {"received":"receive","processed":"process","notified":"notify","applied":"apply",
         "charged":"charge","credited":"credit","approved":"approve","issued":"issue",
         "reviewed":"review","submitted":"submit","exceeded":"exceed","incurred":"incur",
         "took":"take","made":"make","paid":"pay","held":"hold","sent":"send","was":"is"}


def _swap(tokens, i, new_core, sentence):
    lead, core, trail = strip_punct(tokens[i])
    if not core or is_protected(tokens[i], sentence):
        return False
    if core[0].isupper():
        new_core = new_core[0].upper() + new_core[1:]
    tokens[i] = lead + new_core + trail
    return True


def err_agreement(tk, s):
    idx = [i for i, t in enumerate(tk) if strip_punct(t)[1].lower() in AGREEMENT]
    if not idx: return False
    i = random.choice(idx)
    return _swap(tk, i, AGREEMENT[strip_punct(tk[i])[1].lower()], s)

def err_confusion(tk, s):
    idx = [i for i, t in enumerate(tk) if strip_punct(t)[1].lower() in CONFUSIONS]
    if not idx: return False
    i = random.choice(idx)
    return _swap(tk, i, CONFUSIONS[strip_punct(tk[i])[1].lower()], s)

def err_preposition(tk, s):
    idx = [i for i, t in enumerate(tk) if strip_punct(t)[1].lower() in PREPS]
    if not idx: return False
    i = random.choice(idx)
    cur = strip_punct(tk[i])[1].lower()
    return _swap(tk, i, random.choice([p for p in PREPS if p != cur]), s)

def err_tense(tk, s):
    idx = [i for i, t in enumerate(tk) if strip_punct(t)[1].lower() in TENSE]
    if not idx: return False
    i = random.choice(idx)
    return _swap(tk, i, TENSE[strip_punct(tk[i])[1].lower()], s)

def err_article(tk, s):
    arts = [i for i, t in enumerate(tk) if strip_punct(t)[1].lower() in ("a","an","the")]
    if arts and random.random() < 0.55:
        i = random.choice(arts)
        if is_protected(tk[i], s): return False
        if random.random() < 0.5:
            del tk[i]                                   # dropped article
        else:
            cur = strip_punct(tk[i])[1].lower()
            alt = {"a":"an","an":"a","the":"a"}[cur]
            return _swap(tk, i, alt, s)
        return True
    # spurious article
    cand = [i for i in range(1, len(tk)) if free(tk, i, s) and not is_protected(tk[i-1], s)]
    if not cand: return False
    tk.insert(random.choice(cand), "the")
    return True

def err_plural(tk, s):
    cand = [i for i, t in enumerate(tk)
            if len(strip_punct(t)[1]) > 3 and not is_protected(t, s)]
    if not cand: return False
    i = random.choice(cand)
    lead, core, trail = strip_punct(tk[i])
    new = core[:-1] if core.endswith("s") and not core.endswith("ss") else core + "s"
    tk[i] = lead + new + trail
    return True

def err_typo(tk, s):
    cand = [i for i, t in enumerate(tk)
            if len(strip_punct(t)[1]) >= 6 and not is_protected(t, s)]
    if not cand: return False
    i = random.choice(cand)
    lead, core, trail = strip_punct(tk[i])
    j = random.randrange(1, len(core) - 2)
    mode = random.random()
    if mode < 0.45:
        core = core[:j] + core[j+1] + core[j] + core[j+2:]      # transposition
    elif mode < 0.75:
        core = core[:j] + core[j] + core[j:]                    # doubled letter
    else:
        core = core[:j] + core[j+1:]                            # dropped letter
    tk[i] = lead + core + trail
    return True

def err_punct(tk, s):
    commas = [i for i, t in enumerate(tk) if t.endswith(",")]
    if commas and random.random() < 0.5:
        i = random.choice(commas)
        tk[i] = tk[i][:-1]                                      # dropped comma
        return True
    cand = [i for i in range(2, len(tk) - 2)
            if not tk[i].endswith((",", ".", ";")) and free(tk, i, s)]
    if not cand: return False
    i = random.choice(cand)
    tk[i] = tk[i] + ","                                         # spurious comma
    return True

def err_capital(tk, s):
    if len(tk) < 3: return False
    lead, core, trail = strip_punct(tk[0])
    if not core or not core[0].isupper() or is_protected(tk[0], s):
        return False
    tk[0] = lead + core[0].lower() + core[1:] + trail
    return True


# Weights follow MEASURED per-type recall AND measured SENSITIVITY to weight.
# Two runs give a slope for each type (delta recall / delta weight):
#
#   type              w1 -> w2      recall          sensitivity
#   err_confusion   0.16 -> 0.08   0.883 -> 0.883        0.00   free to cut
#   err_agreement   0.20 -> 0.10   0.867 -> 0.817        0.50   cheap to cut
#   err_punct       0.05 -> 0.12   0.650 -> 0.733        1.19
#   err_tense       0.11 -> 0.20   0.517 -> 0.667        1.67
#   err_typo        0.07 -> 0.14   0.633 -> 0.833        2.86
#   err_article     0.15 -> 0.10   0.683 -> 0.517        3.32   expensive to cut
#   err_preposition 0.15 -> 0.18   0.600 -> 0.750        5.00   do not touch
#   err_capital     0.02 -> 0.01   0.933 -> 0.883        5.00   needs a floor
#
# Round 1 treated 0.683 as "good enough" and starved err_article; it is now the
# WORST type at 0.517. Round 2 takes the budget almost entirely from the two
# types proven insensitive to it (confusion, agreement), leaves the responsive
# winners alone, and stops starving err_capital -- 0.01 was below its floor.
ERRORS = [(err_tense, 0.20), (err_article, 0.19), (err_preposition, 0.18),
          (err_punct, 0.12), (err_typo, 0.11), (err_agreement, 0.06),
          (err_plural, 0.06), (err_confusion, 0.05), (err_capital, 0.03)]

def corrupt(sentence, max_errors=2):
    fns = [f for f, _ in ERRORS]
    wts = [w for _, w in ERRORS]
    tk = toks(sentence)
    applied = 0
    for _ in range(random.randint(1, max_errors) * 3):
        if applied >= max_errors:
            break
        fn = random.choices(fns, weights=wts, k=1)[0]
        if fn(tk, sentence):
            applied += 1
    out = " ".join(tk)
    return out if applied and out != sentence else None

for s in random.sample(BANK_SENTS, min(5, len(BANK_SENTS))):
    c = corrupt(s)
    if c:
        print("✗", c)
        print("✓", s)
        print(" ", json.dumps(roundtrips(c, s), ensure_ascii=False), "\n")

## 8. Assemble training data — Spark for union, dedupe, split, and Parquet

We generate the bounded synthetic rows in Python, then hand the resulting records to Spark for the relational data-engineering stage.

LLaMA-Factory supports Parquet, so we do **not** collapse the Spark output into one giant JSON array.


In [ ]:
stage("assembling records")

def make_general_records():
    out = []

    # Tagged by provenance so the dedupe report shows how much of the mix is
    # real human annotation versus scripted corruption.
    for tag, pairs in (("coedit", coedit_pairs),
                       ("c4", c4_pairs),
                       ("wi_locness", WI_PAIRS),
                       ("lang8", LANG8_PAIRS)):
        for s, t, e in pairs:
            out.append(to_alpaca(s, e, tag))

    general_clean = ([t for _, t, _ in coedit_pairs]
                     + [t for _, t, _ in c4_pairs]
                     + [t for _, t, _ in WI_PAIRS]
                     + [t for _, t, _ in LANG8_PAIRS])
    random.shuffle(general_clean)

    n_noop = int(len(out) * NOOP_RATIO_GEN)
    for t in general_clean[:n_noop]:
        out.append(to_alpaca(t, [], "noop_general"))

    return out


def make_banking_records():
    out = []

    if not BANK_SENTS:
        return out

    bank_pool = BANK_SENTS * (N_BANK_NOOP // len(BANK_SENTS) + 1)
    random.shuffle(bank_pool)

    for t in bank_pool[:N_BANK_NOOP]:
        out.append(to_alpaca(t, [], "noop_bank"))

    # MEASURED BUG: corrupt() defaults to max_errors=2, so every banking
    # example carried at most 2 edits. Once banking grew to ~49% of the
    # correction examples, that taught the model that sentences have 1-2
    # errors. Per-edit recall on general text collapsed exactly at the
    # boundary -- 0.679 at 2 gold edits, 0.203 at 3 -- and exact_match on
    # 3+ edit sentences was zero across 85 cases. The banking eval could
    # never reveal it, because the eval used the same capped corrupt().
    made, tries = 0, 0
    while made < N_BANK_GEC and tries < N_BANK_GEC * 25:
        tries += 1
        t = random.choice(BANK_SENTS)
        c = corrupt(t, max_errors=random.choice([1, 2, 2, 3, 3, 4]))

        if not c:
            continue

        e = roundtrips(c, t)
        if e is None or len(e) > 4:
            continue

        out.append(to_alpaca(c, e, "gec_bank"))
        made += 1

    return out


records = make_general_records() + make_banking_records()
random.shuffle(records)

print(Counter(r["_tag"] for r in records))
noop = sum(r["output"] == "[]" for r in records)
print(f"Total before Spark dedupe: {len(records):,}")
print(f"No-op share: {noop / max(len(records), 1):.1%}")


In [ ]:
stage("spark dedupe + parquet write")

records_df = spark.createDataFrame(records)

clean_df = (
    records_df
    .dropDuplicates(["input", "output"])
    .select("instruction", "input", "output", "_tag")
).cache()

print("After dedupe:", clean_df.count())

clean_df.groupBy("_tag").count().orderBy(F.desc("count")).show()

bucket = F.pmod(F.xxhash64("input", "output"), F.lit(100))
train_df = clean_df.filter(bucket < 98).drop("_tag")
dev_df   = clean_df.filter(bucket >= 98).drop("_tag")


def write_parquet_dir(df, path):
    """Write a Spark parquet directory that LLaMA-Factory can actually load.

    Spark emits `_SUCCESS` plus hidden `.crc` sidecars next to the part files.
    LLaMA-Factory's loader does a plain os.listdir() on the directory and raises
    "File types should be identical." on anything that is not .parquet, so the
    sidecars have to go.
    """
    df.write.mode("overwrite").parquet(str(path))

    for p in Path(path).iterdir():
        if p.is_file() and (p.name.startswith(".") or p.suffix != ".parquet"):
            p.unlink()

    parts = sorted(Path(path).glob("*.parquet"))
    if not parts:
        raise RuntimeError(f"No parquet part files were written to {path}")
    return parts


TRAIN_PARQUET = DATA_DIR / "bank_gec_train.parquet"
DEV_PARQUET   = DATA_DIR / "bank_gec_dev.parquet"

train_parts = write_parquet_dir(train_df, TRAIN_PARQUET)
dev_parts   = write_parquet_dir(dev_df, DEV_PARQUET)

print("train:", train_df.count(), f"({len(train_parts)} parquet parts)")
print("dev:", dev_df.count(), f"({len(dev_parts)} parquet parts)")
print("train path:", TRAIN_PARQUET)
print("dev path:", DEV_PARQUET)

In [ ]:
info_path = LF_DIR / "data" / "dataset_info.json"
info = json.loads(info_path.read_text())

for name, path in [
    ("bank_gec_train", TRAIN_PARQUET),
    ("bank_gec_dev", DEV_PARQUET),
]:
    info[name] = {
        "file_name": str(path.resolve()),
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
        },
    }

info_path.write_text(
    json.dumps(info, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Small dev split only: safe to collect for Transformers-based evaluation.
dev = dev_df.toPandas().to_dict("records")
print("dev rows collected:", len(dev))
display(pd.DataFrame(dev).head(3))


## 9. Baseline — scored per domain, not blended

The first run reported a single blended F0.5 and concluded the fine-tune lost.
That eval set was ~95% general web text, so it described a domain this model was
never built for. Re-scored per domain, the same checkpoint returned **F0.5 0.639
on banking vs 0.399 for coedit-large**, while general text stayed near-tied.

Three things changed here:

- **Banking and general are reported separately.** Blending them hid the result.
- **Predictions are canonicalised** before scoring. The baseline's text output is
  converted with `derive_edits` — the same function that made the gold — so it
  was being handed difflib's segmentation for free while the model had to guess
  it. Both sides are now normalised.
- **`exact_match` is reported**: does applying the edits actually reproduce the
  target sentence? It is the only number production cares about.

Banking eval sentences are held out in section 6 before any of them can reach
the training set.


In [ ]:
# span-level scoring. F0.5 weights precision 2x — the right choice for a checker.
def score_edits(gold, pred):
    g = {(e["o"], e["c"]) for e in gold}
    p = {(e["o"], e["c"]) for e in pred}
    return len(g & p), len(p - g), len(g - p)      # tp, fp, fn

def prf(tp, fp, fn, beta=0.5):
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    b2 = beta ** 2
    F = ((1 + b2) * P * R / (b2 * P + R)) if (P and R) else 0.0
    return P, R, F

def norm(s):
    return re.sub(r"\s+", " ", s).strip()

def sane(src_text, edits):
    """Drop malformed edits and spans that are not actually in the source."""
    return [e for e in edits
            if isinstance(e, dict) and e.get("o") and e["o"] in src_text]

def canonical(src_text, edits):
    """Re-express edits in difflib's segmentation — the one the gold uses.

    The baseline gets this for free: its text output is converted with
    derive_edits, the same function that produced the gold. Without this the
    model is scored on whether it guessed difflib's span boundaries.
    """
    edits = sane(src_text, edits)
    if not edits:
        return []
    fixed = apply_edits(src_text, edits)
    if fixed is None:
        return []
    return derive_edits(src_text, fixed) or []

def apply_edits_lenient(src_text, edits):
    """Apply what resolves; skip what does not. For inference and production.

    `apply_edits` is all-or-nothing by design -- correct when *building* gold,
    because an edit list that cannot round-trip should be rejected outright.
    It is wrong at inference: the model emits `n` computed against the original
    sentence, but each applied edit mutates the string, so a later index can
    stop resolving and take every good edit in the list down with it.

        src   : "I is told ... and I have on make another payment."
        edits : [{"o":"is","c":"was","n":1}, {"o":"on","c":"to","n":2}]
        strict: None            <- the valid is->was edit is discarded too
        lenient: "I was told ... and I have to make another payment."
    """
    out, used = src_text, 0
    for e in sane(src_text, edits):
        o, c = e["o"], e.get("c", "")
        try:
            n = max(1, int(e.get("n", 1)))
        except (TypeError, ValueError):
            n = 1
        idx, start = -1, 0
        for _ in range(n):
            idx = out.find(o, start)
            if idx == -1:
                break
            start = idx + 1
        if idx == -1:
            idx = out.find(o)          # index drifted: fall back to first hit
            if idx == -1:
                continue                # span genuinely gone: skip this edit only
        out = out[:idx] + c + out[idx + len(o):]
        used += 1
    return norm(out), used


def applied_text(src_text, edits):
    """The only question production cares about: what sentence comes out?"""
    return apply_edits_lenient(src_text, edits)[0]


def corrupt_one(sentence, fn, attempts=6):
    """Inject exactly one error of a known type, so recall can be attributed."""
    for _ in range(attempts):
        tk = toks(sentence)
        if fn(tk, sentence):
            out = " ".join(tk)
            if out != sentence:
                return out
    return None


# ---- eval sets: banking is scored SEPARATELY from general ----------------
# The previous eval mixed them, and the mix was ~95% general web text, so the
# headline number described a domain this model was not built for.
N_EVAL = 60 if SMOKE else 400

# Banking gold is the exact inverse of an injected error — no stylistic
# ambiguity, unlike CoEdIT/C4 labels which often rewrite rather than correct.
EVAL_BANK_GEC, tries = [], 0
while len(EVAL_BANK_GEC) < N_EVAL and tries < N_EVAL * 40:
    tries += 1
    tgt = random.choice(BANK_EVAL_SENTS)
    bad = corrupt(tgt, max_errors=random.choice([1, 2, 2, 3, 3, 4]))
    if not bad:
        continue
    g = roundtrips(bad, tgt)
    if g is None or len(g) > 4:
        continue
    EVAL_BANK_GEC.append((bad, g, norm(tgt)))

EVAL_BANK_NOOP = random.sample(BANK_EVAL_SENTS, min(N_EVAL, len(BANK_EVAL_SENTS)))

EVAL_GEN_GEC = []
for r in dev:
    if r["output"] == "[]":
        continue
    g = json.loads(r["output"])
    t = apply_edits(r["input"], g)
    if t is None:
        continue
    EVAL_GEN_GEC.append((r["input"], g, norm(t)))
    if len(EVAL_GEN_GEC) >= N_EVAL:
        break

EVAL_GEN_NOOP = [r["input"] for r in dev if r["output"] == "[]"][:N_EVAL]

# Per-error-type set: recall aggregated over all types says the model misses
# things, but not WHICH things. This says where to add training data.
N_PER_TYPE = 10 if SMOKE else 60
EVAL_BY_TYPE = {}
for fn, _w in ERRORS:
    got, tries = [], 0
    while len(got) < N_PER_TYPE and tries < N_PER_TYPE * 60:
        tries += 1
        tgt = random.choice(BANK_EVAL_SENTS)
        bad = corrupt_one(tgt, fn)
        if not bad:
            continue
        gd = roundtrips(bad, tgt)
        if gd is None or len(gd) > 4:
            continue
        got.append((bad, gd, norm(tgt)))
    EVAL_BY_TYPE[fn.__name__] = got

print(f"banking eval: {len(EVAL_BANK_GEC)} gec | {len(EVAL_BANK_NOOP)} noop")
print("per-type eval:", {k: len(v) for k, v in EVAL_BY_TYPE.items()})
print(f"general eval: {len(EVAL_GEN_GEC)} gec | {len(EVAL_GEN_NOOP)} noop")

In [ ]:
stage("baseline eval (coedit-large)")

from transformers import AutoTokenizer, T5ForConditionalGeneration

bl_tok = AutoTokenizer.from_pretrained("grammarly/coedit-large")
bl_model = T5ForConditionalGeneration.from_pretrained("grammarly/coedit-large")
if torch.cuda.is_available():
    bl_model = bl_model.cuda().half()
bl_model.eval()

@torch.no_grad()
def coedit_fix(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        batch = ["Fix grammatical errors in this sentence: " + t for t in texts[i:i+bs]]
        enc = bl_tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=256)
        enc = {k: v.to(bl_model.device) for k, v in enc.items()}
        gen = bl_model.generate(**enc, max_length=256, num_beams=1)
        out += bl_tok.batch_decode(gen, skip_special_tokens=True)
    return out


def eval_baseline_on(pairs, noop_pool):
    srcs = [s for s, _, _ in pairs]
    fixed = coedit_fix(srcs)

    acc, exact = [0, 0, 0], 0
    for (s, gold, tgt), f in zip(pairs, fixed):
        # derive_edits IS the gold format, so the baseline is canonical already.
        p = derive_edits(s, f) or []
        a, b, c = score_edits(gold, p)
        acc[0] += a; acc[1] += b; acc[2] += c
        exact += (norm(f) == tgt)
    P, R, F = prf(*acc)

    out = coedit_fix(noop_pool)
    touched = sum(1 for a, b in zip(noop_pool, out) if norm(a) != norm(b))

    return {"P_raw": P, "F0.5_raw": F, "P_canon": P, "R_canon": R, "F0.5_canon": F,
            "exact_match": exact / len(pairs), "json_valid": float("nan"),
            "noop_rate": touched / len(noop_pool)}


BASELINE_BANK = eval_baseline_on(EVAL_BANK_GEC, EVAL_BANK_NOOP)
BASELINE_GEN  = eval_baseline_on(EVAL_GEN_GEC,  EVAL_GEN_NOOP)

print("banking:", json.dumps({k: round(v, 3) for k, v in BASELINE_BANK.items()}))
print("general:", json.dumps({k: round(v, 3) for k, v in BASELINE_GEN.items()}))

## 10. Fine-tune Qwen3-0.6B on one T4

For T4 we use **FP16**, not BF16, and explicitly disable FlashAttention-2.

**This trains on a single GPU, not 2×T4.** On Kaggle's current image (torch 2.10,
transformers 5.0, LLaMA-Factory 0.9.6.dev0) 2-GPU DDP deadlocks in the first
backward pass and never emits a training step — with and without
`ddp_find_unused_parameters`. The same 30 steps that hang for 45 minutes on two
GPUs finish in 84 seconds on one. `TRAIN_GPUS` in the previous cell is the knob
to re-test this when the image changes.

Two consequences, both measured rather than assumed:

- **Optimizer.** 32-bit Adam needs ~13.5 GiB for a full fine-tune of this model
  (fp32 master 3.0 + grads 3.0 + two moments 6.0) and OOMs on a 14.56 GiB T4 at
  any batch size. Adafactor's factored second moment frees ~6 GiB.
- **Schedule.** One T4 sustains ~8.2 samples/s at `per_device_train_batch_size=8`.
  Two epochs over ~200k examples would need ~13.6h, past Kaggle's 12h session
  limit, so this runs **1 epoch over the full dataset** (~6.8h).


In [ ]:
import yaml

FULL_FT = True

cfg = {
    "model_name_or_path": BASE_MODEL,
    "trust_remote_code": True,

    "stage": "sft",
    "do_train": True,
    "finetuning_type": "full" if FULL_FT else "lora",

    "dataset": "bank_gec_train",
    "eval_dataset": "bank_gec_dev",

    # llamafactory-cli inherits this notebook's CWD (/kaggle/working), but
    # dataset_info.json lives inside the cloned repo. Without an explicit
    # dataset_dir it looks for ./data/dataset_info.json and dies.
    "dataset_dir": str((LF_DIR / "data").resolve()),
    "template": "qwen3",
    "enable_thinking": False,
    "cutoff_len": 512,
    "overwrite_cache": True,
    "preprocessing_num_workers": min(8, os.cpu_count() or 4),
    "dataloader_num_workers": min(4, os.cpu_count() or 2),

    "output_dir": str((SAVE_DIR / "qwen3-0.6b-bankgec").resolve()),
    "logging_steps": 20,
    "save_steps": 1000,
    "save_total_limit": 2,
    "plot_loss": True,
    "overwrite_output_dir": True,
    "report_to": "none",

    # Adafactor freed ~6 GiB, so the micro-batch no longer has to be tiny. At
    # batch 2 the T4 managed only 5.7 samples/s, which puts the full run past
    # Kaggle's 12h ceiling; 8x2 keeps the effective batch at 16 while giving
    # the GPU enough work per step to be worth the launch overhead.
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "learning_rate": 2.0e-5 if FULL_FT else 1.0e-4,
    "num_train_epochs": 1.0,
    "lr_scheduler_type": "cosine",
    # warmup_steps is set below from the real step count
    # Full fine-tuning Qwen3-0.6B needs ~13.5 GiB with 32-bit Adam (fp32 master
    # weights 3.0 + grads 3.0 + two Adam moments 6.0), which does not fit a
    # 14.56 GiB T4 at any batch size. Adafactor keeps a factored second moment
    # instead of two full-size ones, freeing ~6 GiB, and unlike 8-bit Adam it
    # is built into transformers -- one less moving part in the stack.
    "optim": "adafactor",
    "fp16": True,
    "bf16": False,
    "gradient_checkpointing": True,
    "flash_attn": "disabled",
    # The original 180000000 is ~5.7 years: any rank desync hangs the session
    # until Kaggle's 12h limit kills it. Fail in 30 minutes instead.
    "ddp_timeout": 1800,
    # Qwen3-0.6B declares tie_word_embeddings=true but ships both tensors, so
    # transformers leaves lm_head untied and it receives no gradient. Without
    # this, DDP blocks forever in the first backward waiting to all-reduce it.
    "ddp_find_unused_parameters": True,

    "per_device_eval_batch_size": 8,
    "eval_strategy": "steps",
    "eval_steps": 500,
}

if not FULL_FT:
    cfg.update({
        "lora_rank": 32,
        "lora_alpha": 64,
        "lora_target": "all",
        "lora_dropout": 0.05,
        "use_rslora": True,
    })

if SMOKE:
    # Enough steps to prove the launch path, checkpointing and eval loop work.
    cfg.update({
        "num_train_epochs": 1.0,
        "max_steps": 60,
        "logging_steps": 10,
        "eval_steps": 30,
        "save_steps": 60,
    })

# transformers 5.x dropped `warmup_ratio` (LLaMA-Factory now installs up to
# 5.17), while older releases read a float `warmup_steps` as a step count. An
# integer step count means the same thing in every version, so derive 3% of the
# real schedule from the training parquet.
import math
import pyarrow.parquet as pq

WARMUP_RATIO = 0.03

def parquet_rows(path):
    path = Path(path)
    if not path.is_absolute():
        path = LF_DIR / "data" / path
    files = sorted(path.rglob("*.parquet")) if path.is_dir() else [path]
    return sum(pq.ParquetFile(f).metadata.num_rows for f in files)

_info = json.loads((LF_DIR / "data" / "dataset_info.json").read_text())
n_train = parquet_rows(_info[cfg["dataset"]]["file_name"])
# Single GPU (TRAIN_GPUS = 1 in the launch cell), so no world-size factor.
steps_per_epoch = math.ceil(n_train / (cfg["per_device_train_batch_size"]
                                       * cfg["gradient_accumulation_steps"]))
total_steps = cfg.get("max_steps") or math.ceil(steps_per_epoch * cfg["num_train_epochs"])
cfg["warmup_steps"] = max(1, round(WARMUP_RATIO * total_steps))
print(f"{n_train:,} train rows -> {total_steps:,} optimizer steps, warmup_steps={cfg['warmup_steps']}")

cfg_path = CFG_DIR / "qwen3_bankgec_t4x2.yaml"
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(cfg_path.read_text())

In [ ]:
# torchrun is about to start two training processes on this box. Release the
# Spark JVM heap first — everything downstream works off collected Python
# objects (`dev`, `BANK_SENTS`) and the parquet files already on disk.
try:
    spark.stop()
    print("Spark stopped.")
except Exception as ex:
    print("Spark stop:", ex)

# The coedit-large baseline is still resident on GPU 0 from section 9 and holds
# ~3.3 GiB. torchrun's workers need that back: a full fine-tune of a 0.6B model
# (fp32 master weights + Adam moments) already wants ~11 GiB of a 14.6 GiB T4.
for _name in ("bl_model", "ft_model"):
    if _name in globals():
        try:
            globals()[_name].to("cpu")
        except Exception:
            pass
        del globals()[_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for _i in range(torch.cuda.device_count()):
        free_b, total_b = torch.cuda.mem_get_info(_i)
        print(f"GPU {_i}: {free_b / 1024**3:.2f} GiB free of {total_b / 1024**3:.2f} GiB")

# MEASURED, not assumed: on this Kaggle image (torch 2.10 + transformers 5.0 +
# LLaMA-Factory 0.9.6.dev0) 2-GPU DDP deadlocks in the first backward pass and
# never emits a step, with or without ddp_find_unused_parameters. Single-GPU
# trains the same 30 steps in 84s. Raise this only after re-testing DDP.
TRAIN_GPUS = 1
VISIBLE = ",".join(str(i) for i in range(TRAIN_GPUS))
print(f"Training on {TRAIN_GPUS} GPU(s): CUDA_VISIBLE_DEVICES={VISIBLE}")

In [ ]:
stage("fine-tuning")

import subprocess

# `!llamafactory-cli ...` swallows the exit status: when training failed, the
# notebook sailed on and resurfaced it as a baffling "no model.safetensors"
# error three cells later. Run it as a checked subprocess instead.
env = dict(os.environ)
env["CUDA_VISIBLE_DEVICES"] = VISIBLE
if TRAIN_GPUS > 1:
    env["FORCE_TORCHRUN"] = "1"
else:
    env.pop("FORCE_TORCHRUN", None)
env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# A hung torchrun would otherwise silently consume the full 12h session.
TRAIN_TIMEOUT_S = 25 * 60 if SMOKE else 9 * 60 * 60

try:
    proc = subprocess.run(
        ["llamafactory-cli", "train", str(cfg_path)],
        env=env, timeout=TRAIN_TIMEOUT_S,
    )
except subprocess.TimeoutExpired:
    raise RuntimeError(
        f"Training exceeded {TRAIN_TIMEOUT_S / 60:.0f} min and was killed. "
        "Check the last logged step above for where it stalled."
    )

if proc.returncode != 0:
    raise RuntimeError(
        f"llamafactory-cli train failed with exit code {proc.returncode}. "
        "Scroll up for the rank-level traceback."
    )

print("Training finished.")

In [ ]:
import subprocess

if not FULL_FT:
    merge_cfg = {
        "model_name_or_path": BASE_MODEL,
        "adapter_name_or_path": cfg["output_dir"],
        "template": "qwen3",
        "enable_thinking": False,
        "finetuning_type": "lora",
        "export_dir": str((SAVE_DIR / "qwen3-0.6b-bankgec-merged").resolve()),
        "export_size": 2,
        "export_legacy_format": False,
    }
    mp = CFG_DIR / "merge.yaml"
    mp.write_text(yaml.safe_dump(merge_cfg, sort_keys=False), encoding="utf-8")
    subprocess.run(["llamafactory-cli", "export", str(mp)], check=True)
    FINAL_MODEL = merge_cfg["export_dir"]
else:
    FINAL_MODEL = cfg["output_dir"]

print("Final model:", FINAL_MODEL)


## 11. Evaluate

In [ ]:
from transformers import AutoModelForCausalLM

ft_tok = AutoTokenizer.from_pretrained(FINAL_MODEL)
# device_map="auto" can shard even a 0.6B model across both T4s, making every
# decode step pay a PCIe hop. It fits on one GPU, so pin it there.
ft_model = AutoModelForCausalLM.from_pretrained(
    FINAL_MODEL, dtype=torch.float16,
    device_map={"": 0} if torch.cuda.is_available() else None)
ft_model.eval()
if ft_tok.pad_token_id is None:
    ft_tok.pad_token = ft_tok.eos_token
ft_tok.padding_side = "left"

@torch.no_grad()
def predict(texts, bs=32):
    outs, ntok = [], []
    for i in range(0, len(texts), bs):
        chunk = texts[i:i+bs]
        prompts = [ft_tok.apply_chat_template(
            # LLaMA-Factory joins instruction and input with "\n"
            # (converter.py:100). Measured: "\n\n" scored the same, but match
            # training rather than rely on the model tolerating the difference.
            [{"role": "user", "content": INSTRUCTION + "\n" + t}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False)
            for t in chunk]
        enc = ft_tok(prompts, return_tensors="pt", padding=True,
                     truncation=True, max_length=512).to(ft_model.device)
        gen = ft_model.generate(**enc, max_new_tokens=160, do_sample=False,
                                pad_token_id=ft_tok.pad_token_id)
        new = gen[:, enc["input_ids"].shape[1]:]
        ntok += [int((row != ft_tok.pad_token_id).sum()) for row in new]
        outs += ft_tok.batch_decode(new, skip_special_tokens=True)
    return outs, ntok

def parse(raw):
    """Tolerant JSON parse. Returns (edits, was_valid)."""
    m = re.search(r"\[.*\]", raw, re.S)
    if not m:
        return [], False
    try:
        v = json.loads(m.group(0))
        return (v, True) if isinstance(v, list) else ([], False)
    except json.JSONDecodeError:
        return [], False

In [ ]:
stage("fine-tuned eval")

def eval_ft_on(name, pairs, noop_pool):
    srcs = [s for s, _, _ in pairs]
    raws, ntok = predict(srcs)

    raw_acc, can_acc = [0, 0, 0], [0, 0, 0]
    exact = valid = 0
    preds = []

    for (s, gold, tgt), raw in zip(pairs, raws):
        p, ok = parse(raw)
        valid += ok
        p_raw, p_can = sane(s, p), canonical(s, p)
        for accum, pp in ((raw_acc, p_raw), (can_acc, p_can)):
            a, b, c = score_edits(gold, pp)
            accum[0] += a; accum[1] += b; accum[2] += c
        got = applied_text(s, p)
        exact += (got == tgt)
        preds.append({"set": name, "src": s,
                      "gold": json.dumps(gold, ensure_ascii=False),
                      "pred": json.dumps(p_raw, ensure_ascii=False),
                      "applied": got, "target": tgt, "raw_out": raw})

    Pr, _, Fr = prf(*raw_acc)
    Pc, Rc, Fc = prf(*can_acc)

    # Measure the SAME event as the baseline: did the sentence actually change?
    # Counting "emitted a non-empty edit list" is a different question -- a
    # no-op edit such as {"o":"office.","c":"office."} would be counted as a
    # false positive even though the text is untouched.
    raws_n, _ = predict(noop_pool)
    flagged = sum(1 for s, raw in zip(noop_pool, raws_n)
                  if applied_text(s, parse(raw)[0]) != norm(s))

    return {"P_raw": Pr, "F0.5_raw": Fr, "P_canon": Pc, "R_canon": Rc,
            "F0.5_canon": Fc, "exact_match": exact / len(pairs),
            "json_valid": valid / len(pairs),
            "noop_rate": flagged / len(noop_pool),
            "mean_output_tokens": sum(ntok) / len(ntok)}, preds


FT_BANK, PRED_BANK = eval_ft_on("banking", EVAL_BANK_GEC, EVAL_BANK_NOOP)
FT_GEN,  PRED_GEN  = eval_ft_on("general", EVAL_GEN_GEC,  EVAL_GEN_NOOP)


def table(title, base, ft):
    print("\n" + "=" * 74)
    print(title)
    print("=" * 74)
    print(f"{'metric':<18}{'coedit-large':>16}{'qwen3-0.6b-ft':>18}")
    print("-" * 74)
    for k in ["P_raw", "F0.5_raw", "P_canon", "R_canon", "F0.5_canon",
              "exact_match", "noop_rate", "json_valid"]:
        b = base.get(k, float("nan"))
        bs = "-" if b != b else f"{b:.3f}"
        print(f"{k:<18}{bs:>16}{ft[k]:>18.3f}")

table("BANKING  — the domain this model is for", BASELINE_BANK, FT_BANK)
table("GENERAL  — web text (coedit-large's home turf)", BASELINE_GEN, FT_GEN)

# ---- where does the recall actually go? --------------------------------
# Aggregate recall says the model misses errors; this says which kinds, which
# is the difference between "needs more data" and "needs more data of type X".
print("\n" + "=" * 74)
print("RECALL BY ERROR TYPE  (banking, one injected error each)")
print("=" * 74)
print(f"{'error type':<18}{'n':>5}{'recall':>10}{'exact':>10}")
print("-" * 74)

type_rows = []
for tname, pairs in EVAL_BY_TYPE.items():
    if not pairs:
        continue
    raws, _ = predict([s for s, _, _ in pairs])
    tp = fn_ = 0
    ex = 0
    for (src, gold, tgt), raw in zip(pairs, raws):
        p, _ok = parse(raw)
        a, _b, c = score_edits(gold, canonical(src, p))
        tp += a; fn_ += c
        ex += (applied_text(src, p) == tgt)
    rec = tp / (tp + fn_) if (tp + fn_) else 0.0
    type_rows.append((tname, len(pairs), rec, ex / len(pairs)))

for tname, n, rec, ex in sorted(type_rows, key=lambda r: r[2]):
    print(f"{tname:<18}{n:>5}{rec:>10.3f}{ex:>10.3f}")

# ---- does it degrade as sentences get harder? --------------------------
print("\n" + "=" * 74)
print("BANKING ACCURACY BY NUMBER OF GOLD EDITS")
print("=" * 74)
print(f"{'gold edits':<14}{'n':>6}{'exact':>10}")
print("-" * 74)
by_k = {}
for p in PRED_BANK:
    k = len(json.loads(p["gold"]))
    by_k.setdefault(k, []).append(p["applied"] == p["target"])
for k in sorted(by_k):
    v = by_k[k]
    print(f"{k:<14}{len(v):>6}{sum(v) / len(v):>10.3f}")

pd.DataFrame(PRED_BANK + PRED_GEN).to_csv("predictions.csv", index=False)
print("\nsaved predictions.csv  |  mean output tokens:",
      round(FT_BANK["mean_output_tokens"], 1))

In [ ]:
# Eyeball the failures — always more informative than the aggregate numbers.
# Banking only: that is the domain being shipped.
hit  = [p for p in PRED_BANK if p["applied"] == p["target"]]
miss = [p for p in PRED_BANK if p["applied"] != p["target"]]
print(f"banking: {len(hit)} exact / {len(PRED_BANK)}\n")

print("--- correct ---")
for p in hit[:4]:
    print("SRC :", p["src"])
    print("PRED:", p["pred"], "\n")

print("--- failures ---")
for p in miss[:8]:
    print("SRC :", p["src"])
    print("GOLD:", p["gold"])
    print("PRED:", p["pred"])
    print("GOT :", p["applied"])
    print("WANT:", p["target"], "\n")

## 12. What did Pandas vs Spark teach us?

Use the benchmark table from Section 4 in your write-up.

A good conclusion is not "`Spark is faster`". A stronger conclusion is:

> On a single Kaggle VM, Pandas was competitive/faster for the moderate CoEdIT sample and custom Python `difflib` logic. Spark added overhead on the Python-UDF stage. Spark was then used for the parts that benefit from a DataFrame engine: large CSV/Parquet ingestion, sentence explosion, filtering, corpus profiling, union, deduplication, deterministic split, and Parquet output. This keeps the pipeline scalable without forcing small Python-native workloads into Spark.

### When I would choose each

| Situation | Choice |
|---|---|
| Fits comfortably in RAM, exploratory work, custom Python functions | Pandas |
| Millions+ rows with heavy filter/group/join/dedupe | Spark |
| Larger than one machine's RAM | Spark |
| Real multi-node cluster / object-store data lake | Spark |
| Small bounded synthetic generation | Python/Pandas |
| Final SFT training | PyTorch/LLaMA-Factory, not Spark |


## 13. Serving

For T4, do not expect FlashAttention-2. vLLM can use another compatible attention backend.

```bash
vllm serve gec_bank/saves/qwen3-0.6b-bankgec   --dtype half   --max-model-len 512   --gpu-memory-utilization 0.85   --enforce-eager
```

Keep `enable_thinking=False` consistently during training/inference for this task.
